In [ ]:
import pandas as pd
import plotly.express as px
import geopandas as gpd
import plotly.graph_objects as go
import json
import numpy as np 
from dateutil.parser import parse
import datetime

In [ ]:
df_PM10 = pd.read_csv("../data/geodair/Moy_journ_PM10_20170101_20231128.csv", sep = ";")
df_PM25 = pd.read_csv("../data/geodair/Moy_journ_PM25_20170101_20231128.csv", sep = ";")
df_O3 = pd.read_csv("../data/geodair/Moy_journ_O3_20170101_20231128.csv", sep = ";")
df_NO2_1 = pd.read_csv("../data/geodair/Moy_journ_NO2_20170101_20211231.csv", sep = ";")
df_NO2_2 = pd.read_csv("../data/geodair/Moy_journ_NO2_20220101_20231128.csv", sep = ";")
df_NO2 = pd.concat([df_NO2_1,df_NO2_2])

In [ ]:
col_to_keep = ["Date de début", 'code site','valeur brute']

df_PM10 = df_PM10[col_to_keep]
df_PM25 = df_PM25[col_to_keep]
df_O3 = df_O3[col_to_keep]
df_NO2 = df_NO2[col_to_keep]


In [ ]:
df_PM10['date'] = [parse(date).date() for date in df_PM10["Date de début"]]
df_PM25['date'] = [parse(date).date() for date in df_PM25["Date de début"]]
df_O3['date'] = [parse(date).date() for date in df_O3["Date de début"]]
df_NO2['date'] = [parse(date).date() for date in df_NO2["Date de début"]]

In [ ]:
df_PM10_agg = df_PM10.groupby('date')['valeur brute'].mean().reset_index()
df_PM25_agg = df_PM25.groupby('date')['valeur brute'].mean().reset_index()
df_O3_agg = df_O3.groupby('date')['valeur brute'].mean().reset_index()
df_NO2_agg = df_NO2.groupby('date')['valeur brute'].mean().reset_index()


In [ ]:
df_PM10_agg.rename(columns={'valeur brute': 'valeur_PM10'}, inplace=True)
df_PM25_agg.rename(columns={'valeur brute': 'valeur_PM25'}, inplace=True)
df_O3_agg.rename(columns={'valeur brute': 'valeur_O3'}, inplace=True)
df_NO2_agg.rename(columns={'valeur brute': 'valeur_NO2'}, inplace=True)

In [ ]:
df_PM10_agg

In [ ]:
result = pd.merge(df_PM10_agg, df_PM25_agg, on='date', how='outer')
result_2 = pd.merge(result, df_O3_agg, on='date', how='outer')
df_polluants = pd.merge(result_2, df_NO2_agg, on='date', how='outer')

# Afficher le DataFrame résultant
print(df_polluants)



In [ ]:
df_polluants.to_csv("../data/geodair/Moy_journ_station_polluants_20170101_20231128.csv", sep=";")

In [ ]:
fig = px.line(df_polluants, x="date", y=["valeur_PM10","valeur_PM25"],
              hover_data={"date": "|%B %d, %Y"},
              title='Moyenne journalière des différents polluants pour les stations de mesures présentes en Ile de France')
fig.update_xaxes(
    dtick="M1",
    tickformat="%b\n%Y")
fig.show()

In [ ]:
import netCDF4 as nc
fn = 'c:/Users/jbocque1/Downloads/PREVAIR.analyse.20231029.MAXJ.NO2.public.nc'
ds = nc.Dataset(fn)